# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [3]:
from peft import prepare_model_for_kbit_training

from generation.model import load_base_model
from generation.data import load_train_df
from generation.train import train_adapter
from generation.style_loss import build_style_weights, top_tokens

model, tokenizer = load_base_model()
model = prepare_model_for_kbit_training(model)

train_df = load_train_df()

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Main adapters (LoRA)

In [4]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8 (overwrite=True to force)


'artifacts/adapters/tool_lora_r8'

## Ablation: DoRA (same artist)

In [5]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=True)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=True)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_dora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_dora_r8 (overwrite=True to force)


'artifacts/adapters/tool_dora_r8'

## Ablation: Rank

In [6]:
train_adapter(model, tokenizer, train_df, "Gojira", r=4, use_dora=False)
train_adapter(model, tokenizer, train_df, "Gojira", r=16, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r4 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r16 (overwrite=True to force)


'artifacts/adapters/gojira_lora_r16'

## Ablation: Rank on Tool + Mastodon

Extends the rank sweep beyond Gojira (where r4/r8 sit at ceiling, inseparable).
Picked by hypothesis: Tool = under-committed plain adapter (does r16 capacity
commit it?); Mastodon = largest dataset (does the r16 overfit drop vanish with
~2x data?).

In [ ]:
for artist in ["Tool", "Mastodon"]:
    train_adapter(model, tokenizer, train_df, artist, r=4, use_dora=False, overwrite=False)
    train_adapter(model, tokenizer, train_df, artist, r=16, use_dora=False, overwrite=False)

## Remaining artists (LoRA + DoRA r=8)

Completes the lineup so the attribution table and perplexity diagonal cover all 5 artists, and the LoRA-vs-DoRA ablation generalizes beyond Gojira/Tool.

In [7]:
for artist in ["Death", "Mastodon", "Opeth"]:
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=False, overwrite=False)
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=True, overwrite=False)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
5,3.256268
10,2.633807
15,2.433170
20,2.339525
25,2.277219
30,1.918134
35,1.728648
40,1.680940
45,1.579367
50,1.171535


...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/139 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/139 [00:00<?, ? examples/s]

Step,Training Loss
5,3.588384
10,2.960794
15,2.632822
20,3.181625
25,2.723610
30,2.792417
35,2.972565
40,2.387235
45,2.026444
50,2.131061


Saved: artifacts/adapters/mastodon_dora_r8 (139 songs, dora, r=8)
[skip] adapter exists, not retraining: artifacts/adapters/opeth_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/opeth_dora_r8 (overwrite=True to force)


## Experiment: Style-Weighted Loss

Up-weight artist-distinctive tokens (token-level TF-IDF) in the cross-entropy so
the adapter learns artist vocabulary, not just genre. Building/inspecting the
weights is CPU-only; only the `train_adapter` call below needs the GPU. Trains to
`*_sw` adapters so they can be A/B'd against the plain ones on attribution.

In [8]:
from config import ARTISTS

style_weights = {a: build_style_weights(a, train_df, tokenizer, text_col="clean") for a in ARTISTS}

for artist in ARTISTS:
    top_tokens(style_weights[artist], tokenizer, n=30, label=artist)

Gojira: children, !, star, ved, low, load, else, attention, point, round, present, not,
        Get, code, fight, create, understand, dig, throw, Ind, positive, ind, bag,
        space, protect, understanding, shape, New, structure,
Tool: swim, calling, ., Love, information, hey, cheat, Fuck, shit, shoving, Dr, third,
      press, in, frightened, fucking, dick, pushing, adds, dumb, 1, nard, thinking,
      finger, sober, phone, Jesus, 5, Back, wanna
Death: Cor, Den, gore, ritual, ation, zombie, ots, corpses, icide, deceased, drug, orn,
       killer, bloody, Zombie, escaping, corpse, plug, Evil, emotions, Pain, machine,
       pse, util, magg, cunt, Sac, grinder, ificial, Su
Mastodon: road, ill, Wall, Prote, running, wal, ctor, teeth, danger, vine, crystal,
          arian, dirt, fine, warned, aly, failed, song, pretend, paid, scar, fallen,
          curl, Work, Blue, whale, ash, sheep, fight, rel
Opeth: iled, ...., firm, se, white, April, features, conj, st, Ke, mark, iding, aby,
    

In [9]:
for a in ARTISTS:
    train_adapter(model, tokenizer, train_df, a, r=8, use_dora=False, style_weights=style_weights[a])

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8_sw (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8_sw (overwrite=True to force)


Adding EOS to train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Step,Training Loss
5,3.552558
10,2.549991
15,2.200439
20,2.486421
25,2.156588
30,1.736193
35,1.452167
40,1.440749
45,1.356726
50,1.041541


...

Saved: artifacts/adapters/death_lora_r8_sw (104 songs, lora, r=8_sw)


Adding EOS to train dataset:   0%|          | 0/139 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/139 [00:00<?, ? examples/s]

Step,Training Loss
5,3.930545
10,3.678198
15,2.595032
20,3.067568
25,2.839830
30,2.725879
35,2.937805
40,2.277054
45,1.775106
50,2.045079


Saved: artifacts/adapters/mastodon_lora_r8_sw (139 songs, lora, r=8_sw)
[skip] adapter exists, not retraining: artifacts/adapters/opeth_lora_r8_sw (overwrite=True to force)
